# IMDB Sentiment Training Tutorial (Hugging Face)

This notebook shows how to prepare IMDB for your nano-llm using this format:

```text
<bos>[SENTIMENT] positive [/SENTIMENT] [REVIEW] this movie was wonderful... <eos>
```

It includes:
- data loading from Hugging Face
- prompt formatting
- tokenizer preparation
- dataset construction for next-token prediction
- practical training suggestions for your current project

## 0) Setup

Install extra dependency if needed:

```bash
pip install datasets
```

Your project already uses `tokenizers` and PyTorch.

In [1]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")
sys.path.insert(0, str(repo_root / "src"))

from datasets import load_dataset
from nano_llm.tokenizer import HFByteBPETokenizer

c:\Users\dyh\anaconda3\envs\CausalFlow\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Load IMDB dataset from Hugging Face

In [2]:
ds = load_dataset("imdb")
print(ds)
print("train size:", len(ds["train"]))
print("test size :", len(ds["test"]))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
train size: 25000
test size : 25000


## 2) Format examples with sentiment + review tags

We convert labels to `positive` / `negative` and wrap text in your desired structure.

In [3]:
def label_to_sentiment(label: int) -> str:
    return "positive" if int(label) == 1 else "negative"


def clean_text(text: str) -> str:
    # keep this minimal and deterministic
    return " ".join(text.split())


def format_imdb_example(text: str, label: int) -> str:
    sentiment = label_to_sentiment(label)
    review = clean_text(text)
    return f"<bos>[SENTIMENT] {sentiment} [/SENTIMENT] [REVIEW] {review} <eos>"


formatted_train = [format_imdb_example(x["text"], x["label"]) for x in ds["train"]]
formatted_test = [format_imdb_example(x["text"], x["label"]) for x in ds["test"]]

print(formatted_train[0][:300])
print("...")

<bos>[SENTIMENT] negative [/SENTIMENT] [REVIEW] I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fa
...


## 3) Train tokenizer on formatted train text

Use `HFByteBPETokenizer` for a modern byte-level BPE pipeline.

In [4]:
# Join with newlines so each sample remains naturally separated.
tokenizer_train_text = "\n".join(formatted_train)

tok = HFByteBPETokenizer.from_text(
    tokenizer_train_text,
    vocab_size=8000,
    word_boundary_aware=False,
)

print("tokenizer type:", tok.to_state()["type"])
print("vocab size:", tok.vocab_size)

sample = formatted_train[0]
ids = tok.encode(sample)
print("sample token count:", len(ids))
print("roundtrip ok:", tok.decode(ids) == sample)

tokenizer type: hf_bpe_byte
vocab size: 8000
sample token count: 411
roundtrip ok: False


## 4) Build train/val text blobs for your current training loop

Your `nano_llm.train` currently expects one `train_text` and one `val_text` string.
You can concatenate formatted examples with newlines.

In [5]:
# A practical split for LM fine-tuning in this project:
# - Use IMDB train split for training text
# - Use IMDB test split for validation text (or carve from train)
train_text_blob = "\n".join(formatted_train)
val_text_blob = "\n".join(formatted_test)

print("train chars:", len(train_text_blob))
print("val chars  :", len(val_text_blob))

train chars: 34501242
val chars  : 33719228


## 5) Suggestions for model/training settings

For IMDB with this prompt style, start with:

- `tokenizer_type`: `hf_bpe_byte`
- `bpe_vocab_size`: `8000` (or 4000 if memory-limited)
- `seq_len`: `256` (go `384`/`512` only if memory allows)
- `d_model`: `128` or `256`
- `num_layers`: `4-6`
- `batch_size`: as high as fits GPU memory
- `learning_rate`: `2e-4` to `6e-4`
- `early_stopping_patience`: `3-5`

Why these settings:
- IMDB reviews are longer than Tiny Shakespeare lines.
- Sentiment marker tokens (`[SENTIMENT] ...`) benefit from stable context windows.
- Byte-level BPE gives robust handling of punctuation and Unicode artifacts.

## 6) How to integrate into current codebase (minimal plan)

Your current `load_tiny_shakespeare()` is hardcoded. Minimal clean extension:

1. Add a new loader in `src/nano_llm/data.py`, e.g. `load_imdb_sentiment_format(...)`.
2. In `src/nano_llm/train.py`, switch by `dataset_id`:
   - `tiny_shakespeare` -> existing loader
   - `imdb_sentiment` -> new IMDB loader
3. Keep the rest of the training pipeline unchanged (tokenizer + dataloaders already support this).

Optional: add `scripts/train.py --dataset-id imdb_sentiment` for CLI control.

## 7) Inference prompt examples

Generation prompt should follow training format:

```text
<bos>[SENTIMENT] positive [/SENTIMENT] [REVIEW]
```

or

```text
<bos>[SENTIMENT] negative [/SENTIMENT] [REVIEW]
```

Then let the model continue the review text and stop at `<eos>`.